In [1]:
import os, json, joblib
import numpy as np
from sklearn.preprocessing import StandardScaler

X = np.load("../data/processed/X_all.npy")
C = np.load("../data/processed/C_all.npy")

# shuffle split
rng = np.random.default_rng(42)
idx = rng.permutation(len(X))
n_val = int(0.15 * len(X))
val_idx = idx[:n_val]
tr_idx  = idx[n_val:]

X_tr, C_tr = X[tr_idx], C[tr_idx]
X_val, C_val = X[val_idx], C[val_idx]

# scaler for X only
scaler = StandardScaler()
N,T,F = X_tr.shape
scaler.fit(X_tr.reshape(N*T, F))

def apply_scaler(sc, X):
    N,T,F = X.shape
    return sc.transform(X.reshape(N*T, F)).reshape(N,T,F).astype(np.float32)

X_trs  = apply_scaler(scaler, X_tr)
X_vals = apply_scaler(scaler, X_val)

In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

@keras.utils.register_keras_serializable(package="Custom")
class FiLM(layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.modulation = layers.Dense(2 * units)

    def call(self, inputs):
        x, context = inputs
        gamma_beta = self.modulation(context)
        gamma, beta = tf.split(gamma_beta, 2, axis=-1)
        gamma = tf.expand_dims(gamma, axis=1)
        beta  = tf.expand_dims(beta, axis=1)
        return x * (1.0 + gamma) + beta

def build_film_ae(window_size, n_features, context_dim, latent=64):
    x_in = keras.Input(shape=(window_size, n_features), name="x")
    c_in = keras.Input(shape=(context_dim,), name="context")

    h = layers.Conv1D(64, 3, padding="same", activation="relu")(x_in)
    h = FiLM(64)([h, c_in])
    h = layers.Conv1D(64, 3, padding="same", activation="relu")(h)
    h = layers.GlobalAveragePooling1D()(h)
    z = layers.Dense(latent, activation="relu")(h)

    d = layers.Dense(window_size * 64, activation="relu")(z)
    d = layers.Reshape((window_size, 64))(d)
    d = FiLM(64)([d, c_in])
    d = layers.Conv1D(64, 3, padding="same", activation="relu")(d)
    out = layers.Conv1D(n_features, 3, padding="same")(d)

    model = keras.Model([x_in, c_in], out)
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
    return model

WINDOW_SIZE = X_trs.shape[1]
model = build_film_ae(WINDOW_SIZE, X_trs.shape[2], C_tr.shape[1])
model.summary()

2026-02-24 09:41:11.747972: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
W0000 00:00:1771906273.397017  274267 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ x (InputLayer)      │ (None, 24, 8)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 24, 64)    │      1,600 │ x[0][0]           │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ context             │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fi_lm (FiLM)        │ (None, 24, 64)    │        768 │ conv1d[0][0],     │
│                     │                   │            │ context[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 24, 64)    │     12,352 │ fi_lm[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ conv1d_1[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      4,160 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1536)      │     99,840 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 24, 64)    │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fi_lm_1 (FiLM)      │ (None, 24, 64)    │        768 │ reshape[0][0],    │
│                     │                   │            │ context[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 24, 64)    │     12,352 │ fi_lm_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 24, 8)     │      1,544 │ conv1d_2[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 133,384 (521.03 KB)

 Trainable params: 133,384 (521.03 KB)

 Non-trainable params: 0 (0.00 B)

In [3]:
hist = model.fit(
    [X_trs, C_tr], X_trs,
    validation_data=([X_vals, C_val], X_vals),
    epochs=30, batch_size=256
)

Epoch 1/30


2026-02-24 09:41:13.539207: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 34868736 exceeds 10% of free system memory.
2026-02-24 09:41:13.579594: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 34868736 exceeds 10% of free system memory.


178/178 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - loss: 78959.4375 - val_loss: 448.5739
Epoch 2/30
178/178 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 129.0595 - val_loss: 50.1306
Epoch 3/30
178/178 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 32.9603 - val_loss: 23.9447
Epoch 4/30
178/178 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 18.1684 - val_loss: 14.1698
Epoch 5/30
178/178 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 12.0819 - val_loss: 11.4496
Epoch 6/30
178/178 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 9.5195 - val_loss: 8.1664
Epoch 7/30
178/178 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 6.9398 - val_loss: 6.8881
Epoch 8/30
178/178 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 6.2219 - val_loss: 5.9891
Epoch 9/30
178/178 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 5.4175 - val_loss: 6.0414
Epoch 10/30
178/178 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 4.6817 - val_loss: 5.2753
Epoch 11/30
178/178 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 4.5296 - val_loss: 4.8273
Epoch 12/30
178/178 ━━━━━━━━━━━━━━━━━

In [4]:
import numpy as np

TOPK = 50
Q = 0.995

def topk_score(x_true, x_pred, k):
    err = (x_true - x_pred) ** 2
    flat = err.reshape(len(err), -1)
    k = min(k, flat.shape[1])
    part = np.partition(flat, -k, axis=1)[:, -k:]
    return np.mean(part, axis=1)

X_tr_pred = model.predict([X_trs, C_tr], batch_size=256, verbose=0)
tr_scores = topk_score(X_trs, X_tr_pred, TOPK)
THRESHOLD = float(np.quantile(tr_scores, Q))
THRESHOLD

2026-02-24 09:42:21.591605: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 34868736 exceeds 10% of free system memory.
2026-02-24 09:42:22.320051: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 34868736 exceeds 10% of free system memory.


60.53527069091797

In [5]:
import os, joblib
os.makedirs("../saved_model", exist_ok=True)

model.save("../saved_model/ae_model.keras")
joblib.dump(scaler, "../saved_model/scaler.joblib")
joblib.dump({"window_size": WINDOW_SIZE, "topk": TOPK, "threshold": THRESHOLD, "q": Q},
            "../saved_model/detector_meta.joblib")

print("Saved model+scaler+meta")

Saved model+scaler+meta
